In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import networkx as nx
import json

In [ ]:
# Load jsonl file into a list of dictionaries
with open("../data/pred/docs_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

In [ ]:
import random

# Take random subset
random.seed(52)
data = random.sample(data, 1000)

In [ ]:
# Ignore docs with null annotations
data = [d for d in data if d["annotations"]]
len(data)

In [ ]:
from util import extract_ner_labels

b = nx.Graph()

# Create ent-doc pairs
edges = []
for item in data:
    doc_id = (item["id"], "DOC")
    entities = [(e[0].title(), e[1]) for e in extract_ner_labels(item["pred"])]

    for e in entities:
        edges.append((e, doc_id))


b.add_edges_from(edges)

In [ ]:
from networkx.algorithms.minors import contracted_nodes

from Levenshtein import distance
import numpy as np


nodes = list(b.nodes())
sim_matrix = np.full((len(nodes), len(nodes)), np.inf)

for i in range(len(nodes)):
    for j in range(i, len(nodes)):
        if (isinstance(nodes[i][0], int)) or (isinstance(nodes[j][0], int)):
            continue
        else:
            sim_matrix[i, j] = distance(nodes[i][0], nodes[j][0])

m = np.matrix(sim_matrix)

similar_ent = np.argwhere((m <= 1) & (m >= 0))

for i, j in similar_ent:
    if i == j:
        continue
    print(nodes[i], "|", nodes[j], "|", m[i, j])
    try:
        print(f"Degree for node i: {b.degree()[nodes[i]]}")  # type: ignore
        print(f"Degree for node j: {b.degree()[nodes[j]]}")  # type: ignore

        # Contract the two nodes into the node with the higher degree
        if b.degree()[nodes[i]] >= b.degree()[nodes[j]]:  # type: ignore
            b = contracted_nodes(b, nodes[i], nodes[j])
        else:
            b = contracted_nodes(b, nodes[j], nodes[i])

    except KeyError:
        print(f"{nodes[i]} or {nodes[j]} already contracted")

In [ ]:
from util import draw_bipartite

draw_bipartite(b, draw_labels=False, figsize=(10, 5))

In [ ]:
from networkx.algorithms.bipartite.projection import projected_graph
from util import draw_graph

entity_nodes = [n for n in b.nodes() if not isinstance(n[0], int)]
p = projected_graph(b, entity_nodes)

draw_graph(p, title="Graph Projected on Entities", figsize=(12, 9))

In [ ]:
ego = nx.ego_graph(p, ("Enrico Fontana Paolo Conti", "PER"))

draw_graph(
    ego,
    title="Ego Graph",
    figsize=(7, 5),
    node_size=150,
    label_font_size=10,
)

In [ ]:
import pandas as pd
from scipy.stats import hypergeom
from collections import Counter


def get_p_value(df: pd.DataFrame, nodes_num: int):
    return hypergeom.sf(df["n_link"], nodes_num, df["occ_ent_1"], df["occ_ent_2"])


# Use the edges of the projected graph p
edges = list(p.edges())
edges_count = Counter(list(edges))

# n_link: N_ij, number of links between i and j
# occ_ent_1: degree of i
# occ_ent_2: degree of j
df = pd.DataFrame(
    {
        "ent1": [e[0] for e in edges],
        "ent2": [e[1] for e in edges],
        "n_link": [edges_count[e] for e in edges],
        "occ_ent_1": [p.degree[edges[i][0]] for i in range(len(edges))],
        "occ_ent_2": [p.degree[edges[i][1]] for i in range(len(edges))],
    }
)

df["p-value"] = df.apply(lambda x: get_p_value(x, len(p.nodes())), axis=1)
threshold = 0.01 / df.shape[0]
df["significant"] = df["p-value"] < threshold

df

In [ ]:
# Get statistically significant links
tmp = df[["ent1", "ent2"]].loc[df["significant"]]
validated_edges = list(zip(tmp.ent1, tmp.ent2))

val_g = nx.Graph(validated_edges)

draw_graph(val_g, title="Statistically Validated Projected Graph")